# Part C · The two-map activity, on real readers

**Ali Afzal** · BSAI-7B-102 · Data Visualization · Week 5

The same 20 counties, the same boundaries, the same five-class scheme and the same ramp. The **only**
difference between the two maps is the denominator.

In [1]:
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

import geolib as g
from vizlib import PALETTE, save_fig, style_defaults

DATA = Path("data")
CHARTS = Path("charts")
CHARTS.mkdir(exist_ok=True)
style_defaults(dpi=100)
INK, SOFT = PALETTE["ink"], PALETTE["ink_soft"]

cc = pd.read_csv(DATA / "county_crime.csv")
cc["rate"] = cc.crimes_reported / cc.population * 1e5
feats_all = g.read_geojson(DATA / "districts.geojson")

## 1 · Build both maps

In [2]:
geo_codes = {f["code"] for f in feats_all}
unmatched_geo = sorted(geo_codes - set(cc.code))
unmatched_rows = sorted(set(cc.code) - geo_codes)
feats = [f for f in feats_all if f["code"] in set(cc.code)]
print(f"counties: {len(cc)}; polygons: {len(feats_all)}; joined: {len(feats)}")
print(f"polygons with no county (left out): {unmatched_geo}; county rows with no polygon: {unmatched_rows or 'none'}")

counties: 20; polygons: 24; joined: 20
polygons with no county (left out): ['D21', 'D22', 'D23', 'D24']; county rows with no polygon: none


In [3]:
r = cc[["crimes_reported", "population"]].corr().iloc[0, 1]
rank_n = cc.crimes_reported.rank(ascending=False)
rank_r = cc.rate.rank(ascending=False)
moved = (rank_n - rank_r).abs()
print(f"r(crimes_reported, population) = {r:+.3f}")
print("top 3 by count:", cc.nlargest(3, "crimes_reported").county.tolist())
print("top 3 by rate: ", cc.nlargest(3, "rate").county.tolist())
print(f"counties moving 3+ places: {(moved >= 3).sum()} of {len(cc)}")
cc.assign(rank_count=rank_n.astype(int), rank_rate=rank_r.astype(int))[
    ["county", "population", "crimes_reported", "rate", "rank_count", "rank_rate"]].sort_values("rank_rate").round(1)

r(crimes_reported, population) = +0.935
top 3 by count: ['Dunmere', 'Pellbrook', 'Grimsby']
top 3 by rate:  ['Hollowfield', 'Netherby', 'Elmsworth']
counties moving 3+ places: 7 of 20


,county,population,crimes_reported,rate,rank_count,rank_rate
7,Hollowfield,12000,216,1800.0,16,1
12,Netherby,170000,3031,1782.9,4,2
4,Elmsworth,22000,384,1745.5,13,3
11,Marchford,110500,1878,1699.5,6,4
6,Grimsby,187000,3038,1624.6,3,5
8,Ironbridge,122000,1952,1600.0,5,6
14,Pellbrook,358500,5680,1584.4,2,7
18,Thornbury,40000,562,1405.0,9,8
5,Foxhall,66000,840,1272.7,8,9
3,Dunmere,633500,5971,942.5,1,10


**Self-check:** r = +0.935, top three by count Dunmere, Pellbrook, Grimsby, top three by rate Hollowfield,
Netherby, Elmsworth, an overlap of zero, and 7 counties moving three or more places. ✓

In [4]:
lat_ts = float(np.mean([p[1] for f in feats for p in f["ring"]]))
name = dict(zip(cc.code, cc.county))


def draw_map(values, legend_title, fmt, title, path):
    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    edges = g.choropleth(ax, feats, values, "natural", 5, fmt=fmt, legend_title=legend_title)
    for f in feats:
        lon, lat = g.representative_point(f["ring"])
        x, y = g.equal_area_xy([lon], [lat], lat_ts=lat_ts)
        rx, _ = g.equal_area_xy([p[0] for p in f["ring"]], [p[1] for p in f["ring"]], lat_ts=lat_ts)
        narrow = np.ptp(rx) < 140   # km: strip-shaped counties get an upright label
        ax.text(x[0], y[0], name[f["code"]], ha="center", va="center", fontsize=7.5, color=INK,
                rotation=90 if narrow else 0,
                bbox=dict(fc="white", ec="none", alpha=0.7, pad=1))
    ax.set_title(title, loc="center")
    save_fig(fig, CHARTS / path, note=textwrap.fill(
        "20 counties. Five natural (Jenks) classes. Equal-area projection. Four polygons (D21-D24) have no county "
        "and are left out. Source: synthetic course data, CDB601220 Week 5.", 80))
    return fig, edges


fig_a, edges_a = draw_map(dict(zip(cc.code, cc.crimes_reported)), "Crimes reported\n(count)", "{:,.0f}",
                          "Map A: crimes reported", "partC_mapA.png")
fig_b, edges_b = draw_map(dict(zip(cc.code, cc.rate)), "Crimes per 100,000\nresidents", "{:,.0f}",
                          "Map B: crimes per 100,000 residents", "partC_mapB.png")
plt.close("all")
print("class breaks A:", np.round(edges_a).astype(int).tolist())
print("class breaks B:", np.round(edges_b).astype(int).tolist())

class breaks A: [88, 240, 840, 1952, 3038, 5971]
class breaks B: [479, 613, 794, 943, 1405, 1800]


Both maps are saved separately ([`partC_mapA.png`](charts/partC_mapA.png),
[`partC_mapB.png`](charts/partC_mapB.png)) so a reader can be shown one at a time. The titles are neutral
on purpose. A title that states the finding would answer the question for the reader.

## 2 · Run it on at least five readers

One reader at a time, **in this order**: Map A only, "Which three counties would you send extra police to,
and why?"; then Map B, the same question. Answer 1 can't be revised.

1. Set `READER` and `RUN_EXPERIMENT = True`, then run this cell with the reader at the screen.
2. Repeat for each reader. Rows are **appended**, one per reader.
3. Set `RUN_EXPERIMENT = False`, then **Restart & Run All**.

In [5]:
RESULTS = Path("partC_reader_results.csv")
RUN_EXPERIMENT = False
READER = "reader 1"
COUNTIES = {c.lower(): c for c in cc.county}


def ask(path, label):
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    ax.imshow(plt.imread(CHARTS / path))
    ax.axis("off")
    display(fig)
    plt.close(fig)
    while True:
        raw = input(f"{label}: which THREE counties would you send extra police to? (comma-separated) ")
        picks = [COUNTIES.get(p.strip().lower()) for p in raw.split(",")]
        if len(picks) == 3 and all(picks):
            break
        print("Please type exactly three county names from the map.")
    why = input("Why those three? ")
    return picks, why


if RUN_EXPERIMENT:
    a, why_a = ask("partC_mapA.png", "Map A")
    b, why_b = ask("partC_mapB.png", "Map B")
    clear_output()
    row = {"reader": READER, "map_a": "; ".join(a), "why_a": why_a, "map_b": "; ".join(b), "why_b": why_b,
           "overlap": len(set(a) & set(b))}
    pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not RESULTS.exists(), index=False)

if RESULTS.exists():
    results = pd.read_csv(RESULTS)
    display(results)
else:
    results = None
    print("No partC_reader_results.csv yet. Set RUN_EXPERIMENT = True and run this cell once per reader.")

No partC_reader_results.csv yet. Set RUN_EXPERIMENT = True and run this cell once per reader.


## 3 · Report it

In [6]:
top_a = set(cc.nlargest(3, "crimes_reported").county)
top_b = set(cc.nlargest(3, "rate").county)
if results is None:
    print("Waiting for partC_reader_results.csv.")
else:
    if len(results) < 5:
        print(f"Only {len(results)} reader(s) so far; the brief needs at least 5.")
    pa = results.map_a.str.split("; ")
    pb = results.map_b.str.split("; ")
    summary = pd.DataFrame({
        "reader": results.reader,
        "overlap A vs B": results.overlap,
        "A picks that are the top 3 by count": pa.apply(lambda p: len(set(p) & top_a)),
        "B picks that are the top 3 by rate": pb.apply(lambda p: len(set(p) & top_b)),
    })
    display(summary)
    print(f"mean overlap across {len(results)} readers: {results.overlap.mean():.2f} of 3")

Waiting for partC_reader_results.csv.


> **My readers' result** *(fill in from the table)*: __ readers; mean overlap __ of 3. How many readers' Map
> A picks were the three biggest-count counties, and how many readers' Map B picks were the three
> highest-rate counties? If anyone picked the same three on both maps, say what you think happened (did
> they read the legend title?).

**What each map is a map of.** Map A is a map of **where people live**, since r(crimes, population) =
+0.935, so its darkest counties are simply the most populous. Map B is a map of **risk per resident**, and
its darkest counties include Elmsworth (22,000 people) and Hollowfield (12,000), which are the palest
counties on Map A.

**Limits.** Five-odd readers, shown the maps in a fixed order, with no counterbalancing. This shows how
*these* readers behaved. It isn't an estimate for readers in general. Small counties also have unstable
rates (Part B.5), so Hollowfield's top rank on Map B rests on a small denominator and should carry a note.